In [ ]:
from typing import TypedDict, Annotated, final

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, BaseMessage
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.types import interrupt, Command

load_dotenv()

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    approved: bool

In [ ]:
llm = ChatGroq(model="qwen/qwen3-32b", temperature=0.4, reasoning_format="hidden")

search_tool = TavilySearch(
    max_results=3,
    topic="general",
    include_images=False,
    search_depth="advanced"
)

tool_list = [search_tool]
llm_with_tool = llm.bind_tools(tool_list)

In [ ]:
def chat_node(state: ChatState) -> ChatState:
    # print(list(state["messages"]))
    response_llm = llm_with_tool.invoke(input=state["messages"])
    return {"messages": [response_llm]}

In [ ]:
tools_node = ToolNode(tool_list)

In [ ]:
def should_continue(state: ChatState) -> str:
    """Determine if we should continue or end the conversation."""
    messages = state["messages"]
    last_message = messages[-1]

    # If no tools were called, we're done
    if not last_message.tool_calls:
        return END
    else:
        return "ask_human"

In [1]:
# Interactive human approval node can only be used in langgraph node and not in the conditional edge logic. This is because the conditional edge logic is not interactive and cannot wait for human input.
def ask_human(state: ChatState) -> ChatState:
    decision = interrupt({
        "type": "tool_decision",
        "tool_call": state["messages"][-1].tool_calls[0],
        "user_message": "LLM is trying to access the tool, do you want to allow it yes or no?",
    })
    return {"approved": decision["allow"] == "yes"}


def after_asking_human(state: ChatState) -> str:
    # Honor the human's decision: run the tool only if approved.
    return "tools" if state.get("approved", False) else END

NameError: name 'ChatState' is not defined

In [ ]:
checkpointer = InMemorySaver()
graph = StateGraph(ChatState)

graph.add_node("chat_node", chat_node)
graph.add_node("tools", tools_node)
graph.add_node("ask_human", ask_human)

graph.add_edge(START, "chat_node")
graph.add_conditional_edges("chat_node", should_continue, {"ask_human": "ask_human", END: END})
graph.add_conditional_edges("ask_human", after_asking_human, {"tools": "tools", END: END})
graph.add_edge("tools", "chat_node")

workflow = graph.compile(checkpointer=checkpointer)

In [ ]:
workflow

In [ ]:
thread_id = "mychat-ui-4"
config = {"configurable": {"thread_id": thread_id}}

# while True:
user_message = input("Human: ")
initial_state = {"messages": [HumanMessage(content=user_message)]}

print(f"Human: {user_message}")
# if user_message.lower().strip() in ["bye", "exit", "quit"]:
#     break

response = workflow.invoke(input=initial_state, config=config)
if response:
    print(f"AI: {response["messages"][-1].content} \n")

In [ ]:
interrupts = workflow.get_state(config=config).interrupts

if interrupts:
    user_feedback_on_interrupt = input(interrupts[-1].value.get("user_message", ""))
    response_llm =  workflow.invoke(Command(resume={"allow": user_feedback_on_interrupt.lower().strip()}), config=config)
    response_llm = response_llm["messages"][-1].content
    final = "Human rejected the tool use" if not response_llm else response_llm
    print(f"AI: {response_llm} \n")